# 📊 Analyse Exploratoire des Données — Campagnes Marketing
---
**Problématique :** Quels sont les segments de clients les plus rentables en fonction de leurs interactions avec des campagnes marketing ?

**Dataset :** [Marketing Campaign — Kaggle](https://www.kaggle.com/datasets/rodsaldanha/arketingcampaign/data)

**Auteur :** *Votre Nom*  
**Date :** 2024  

---
## Sommaire
1. [Chargement & aperçu des données](#1)
2. [Nettoyage & feature engineering](#2)
3. [Statistiques univariées](#3)
4. [Statistiques multivariées](#4)
5. [Analyse des segments clients](#5)
6. [Performance des campagnes](#6)
7. [Insights & recommandations](#7)


## 1. Chargement & aperçu des données <a id='1'></a>

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# ── Style global ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor"  : "#0d0f1a",
    "axes.facecolor"    : "#1a1d2e",
    "axes.edgecolor"    : "#252840",
    "axes.labelcolor"   : "#c0c8e0",
    "axes.titlecolor"   : "#e0e4ef",
    "axes.titlesize"    : 13,
    "axes.titleweight"  : "600",
    "axes.grid"         : True,
    "grid.color"        : "#252840",
    "grid.linestyle"    : "--",
    "grid.linewidth"    : 0.5,
    "xtick.color"       : "#8892b0",
    "ytick.color"       : "#8892b0",
    "text.color"        : "#c0c8e0",
    "legend.facecolor"  : "#1a1d2e",
    "legend.edgecolor"  : "#252840",
    "legend.labelcolor" : "#c0c8e0",
    "figure.dpi"        : 120,
    "font.family"       : "DejaVu Sans",
})

PALETTE = ["#7b9cff","#64dfdf","#f4a261","#e76f51","#a8dadc","#52b788","#ffd166","#ef476f"]
sns.set_palette(PALETTE)

print("✅ Imports et style chargés avec succès")


In [ ]:
# ── Chargement du dataset ───────────────────────────────────────────────────────
df = pd.read_csv("marketing_campaign.csv", sep=";")

print(f"📐 Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"\n📋 Colonnes :")
for col in df.columns:
    print(f"   • {col} ({df[col].dtype})")


In [ ]:
# ── Aperçu des premières lignes ─────────────────────────────────────────────────
df.head()


In [ ]:
# ── Statistiques descriptives globales ─────────────────────────────────────────
df.describe().T.style.format("{:.2f}") \
    .background_gradient(subset=["mean","std"], cmap="Blues")


In [ ]:
# ── Qualité des données ─────────────────────────────────────────────────────────
print("=" * 50)
print("  AUDIT QUALITÉ DES DONNÉES")
print("=" * 50)

# Valeurs manquantes
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Nulls": nulls, "% manquant": nulls_pct})
missing_df = missing_df[missing_df["Nulls"] > 0]

if missing_df.empty:
    print("\n✅ Aucune valeur manquante détectée")
else:
    print(f"\n⚠️  Variables avec valeurs manquantes :")
    print(missing_df.to_string())

# Doublons
dups = df.duplicated().sum()
print(f"\n{'✅' if dups==0 else '⚠️ '} Doublons : {dups}")

# Valeurs aberrantes dans Marital_Status
print(f"\n📌 Statut marital (valeurs uniques) : {df['Marital_Status'].unique()}")
print(f"📌 Éducation (valeurs uniques) : {df['Education'].unique()}")
print(f"\n📊 Distribution Response (variable cible) :")
print(df["Response"].value_counts().to_string())
print(f"   → Taux de conversion : {df['Response'].mean()*100:.2f}%")


## 2. Nettoyage & Feature Engineering <a id='2'></a>

In [ ]:
# ── Nettoyage ───────────────────────────────────────────────────────────────────
df_clean = df.copy()

# 1. Remplir les 24 valeurs manquantes d'Income par la médiane
median_income = df_clean["Income"].median()
df_clean["Income"] = df_clean["Income"].fillna(median_income)
print(f"✅ Income : 24 nulls remplacés par la médiane ({median_income:,.0f} €)")

# 2. Nettoyer les statuts maritaux aberrants
df_clean["Marital_Clean"] = df_clean["Marital_Status"].replace(
    {"Alone": "Single", "Absurd": "Other", "YOLO": "Other"})
print(f"✅ Marital_Status : 'Alone' → 'Single', 'Absurd'/'YOLO' → 'Other'")

# 3. Convertir Dt_Customer en datetime
df_clean["Dt_Customer"] = pd.to_datetime(df_clean["Dt_Customer"])
print(f"✅ Dt_Customer converti en datetime")

print(f"\n📐 Shape après nettoyage : {df_clean.shape}")


In [ ]:
# ── Feature Engineering ─────────────────────────────────────────────────────────
# Nouvelles variables dérivées

df_clean["Age"]           = datetime.now().year - df_clean["Year_Birth"]
df_clean["TotalSpend"]    = df_clean[["MntWines","MntFruits","MntMeatProducts",
                                       "MntFishProducts","MntSweetProducts","MntGoldProds"]].sum(axis=1)
df_clean["TotalPurchases"]= df_clean[["NumDealsPurchases","NumWebPurchases",
                                       "NumCatalogPurchases","NumStorePurchases"]].sum(axis=1)
df_clean["TotalCampaigns"]= df_clean[["AcceptedCmp1","AcceptedCmp2","AcceptedCmp3",
                                       "AcceptedCmp4","AcceptedCmp5"]].sum(axis=1)
df_clean["HasChildren"]   = (df_clean["Kidhome"] + df_clean["Teenhome"]).clip(upper=1)
df_clean["Seniority"]     = (pd.Timestamp.now() - df_clean["Dt_Customer"]).dt.days // 30

df_clean["Age_Group"]    = pd.cut(df_clean["Age"],
    bins=[17,30,45,60,100], labels=["18–30","31–45","46–60","60+"])
df_clean["Income_Group"] = pd.cut(df_clean["Income"],
    bins=[0,30000,60000,90000,200000], labels=["<30k","30–60k","60–90k",">90k"])

new_vars = ["Age","TotalSpend","TotalPurchases","TotalCampaigns",
            "HasChildren","Seniority","Age_Group","Income_Group"]
print("✅ Nouvelles variables créées :")
for v in new_vars:
    print(f"   • {v}")

df_clean[new_vars].describe().T


## 3. Statistiques Univariées <a id='3'></a>

> Analyse variable par variable — distribution, outliers, asymétrie.

In [ ]:
# ── Distribution des variables numériques clés ──────────────────────────────────
num_vars = ["Age","Income","TotalSpend","TotalPurchases","Recency","Seniority"]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle("Distribution des variables numériques clés", fontsize=15, fontweight="600", y=1.01)

for ax, var in zip(axes.flat, num_vars):
    data = df_clean[var].dropna()
    ax.hist(data, bins=40, color=PALETTE[0], alpha=0.85, edgecolor="#0d0f1a", linewidth=0.3)
    ax.axvline(data.mean(),   color=PALETTE[2], linestyle="--", linewidth=1.4,
               label=f"Moyenne: {data.mean():.0f}")
    ax.axvline(data.median(), color=PALETTE[3], linestyle=":",  linewidth=1.4,
               label=f"Médiane: {data.median():.0f}")
    ax.set_title(var)
    ax.set_xlabel(var); ax.set_ylabel("Fréquence")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("plots/01_distributions_numeriques.png", bbox_inches="tight", dpi=150)
plt.show()
print("💾 Graphique sauvegardé : plots/01_distributions_numeriques.png")


In [ ]:
# ── Skewness & Kurtosis ─────────────────────────────────────────────────────────
num_cols_all = ["Age","Income","TotalSpend","TotalPurchases","Recency","Seniority",
                "MntWines","MntFruits","MntMeatProducts","MntFishProducts",
                "MntSweetProducts","MntGoldProds"]

stats_df = pd.DataFrame({
    "Moyenne"  : df_clean[num_cols_all].mean().round(2),
    "Médiane"  : df_clean[num_cols_all].median().round(2),
    "Std"      : df_clean[num_cols_all].std().round(2),
    "Skewness" : df_clean[num_cols_all].skew().round(3),
    "Kurtosis" : df_clean[num_cols_all].kurtosis().round(3),
})

stats_df["Interprétation"] = stats_df["Skewness"].apply(lambda s:
    "Forte asymétrie droite ⚠️" if s > 1 else
    "Forte asymétrie gauche ⚠️" if s < -1 else
    "Asymétrie modérée" if abs(s) > 0.5 else
    "Distribution symétrique ✅")

stats_df.style.format({
    "Moyenne":"{:.2f}", "Médiane":"{:.2f}", "Std":"{:.2f}",
    "Skewness":"{:.3f}", "Kurtosis":"{:.3f}"
}).background_gradient(subset=["Skewness"], cmap="RdYlGn_r")


In [ ]:
# ── Boxplots — Détection des outliers ──────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Boxplots — Détection des valeurs aberrantes", fontsize=14, fontweight="600")

spend_vars = ["MntWines","MntFruits","MntMeatProducts",
              "MntFishProducts","MntSweetProducts","MntGoldProds"]

for ax, var in zip(axes.flat, spend_vars):
    bp = ax.boxplot(df_clean[var].dropna(),
                    patch_artist=True,
                    medianprops=dict(color=PALETTE[2], linewidth=2.5),
                    boxprops=dict(facecolor=PALETTE[0], alpha=0.55, linewidth=0.8),
                    whiskerprops=dict(color="#c0c8e0", linewidth=0.8),
                    capprops=dict(color="#c0c8e0", linewidth=0.8),
                    flierprops=dict(marker="o", color=PALETTE[3], alpha=0.4, markersize=3))
    ax.set_title(var.replace("Mnt","Dépenses — "))
    ax.set_ylabel("Montant (€)")

plt.tight_layout()
plt.savefig("plots/02_boxplots_outliers.png", bbox_inches="tight", dpi=150)
plt.show()


In [ ]:
# ── Variables catégorielles ─────────────────────────────────────────────────────
cat_vars = ["Education","Marital_Clean","Age_Group","Income_Group","HasChildren"]

fig, axes = plt.subplots(1, len(cat_vars), figsize=(18, 5))
fig.suptitle("Répartition des variables catégorielles", fontsize=14, fontweight="600")

for ax, var in zip(axes, cat_vars):
    counts = df_clean[var].value_counts()
    wedges, texts, autotexts = ax.pie(
        counts.values,
        labels=counts.index.astype(str),
        colors=PALETTE[:len(counts)],
        autopct="%1.1f%%",
        startangle=140,
        pctdistance=0.82,
        wedgeprops=dict(linewidth=1.5, edgecolor="#0d0f1a"))
    for t in texts:    t.set_fontsize(8); t.set_color("#c0c8e0")
    for at in autotexts: at.set_fontsize(7); at.set_color("#0d0f1a"); at.set_fontweight("600")
    ax.set_title(var, fontsize=10)

plt.tight_layout()
plt.savefig("plots/03_variables_categorielles.png", bbox_inches="tight", dpi=150)
plt.show()


## 4. Statistiques Multivariées <a id='4'></a>

> Relations entre variables — corrélations, interactions, effets croisés.

In [ ]:
# ── Matrice de corrélation ──────────────────────────────────────────────────────
corr_cols = ["Age","Income","TotalSpend","TotalPurchases","Recency",
             "MntWines","MntMeatProducts","MntFruits","MntGoldProds",
             "NumWebVisitsMonth","TotalCampaigns","Response"]

corr = df_clean[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)

sns.heatmap(corr, mask=mask, cmap=cmap, center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f",
            annot_kws={"size": 8, "color": "#c0c8e0"},
            linewidths=0.5, linecolor="#252840", ax=ax,
            cbar_kws={"shrink": 0.7})

ax.set_title("Matrice de corrélations — Variables numériques", fontsize=13, pad=14)
plt.tight_layout()
plt.savefig("plots/04_heatmap_correlations.png", bbox_inches="tight", dpi=150)
plt.show()


In [ ]:
# ── Corrélations avec la variable cible (Response) ─────────────────────────────
resp_corr = corr["Response"].drop("Response").sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
colors_bar = [PALETTE[0] if v > 0 else PALETTE[3] for v in resp_corr.values]
bars = ax.bar(resp_corr.index, resp_corr.values, color=colors_bar,
              edgecolor="#0d0f1a", linewidth=0.4, alpha=0.9)
ax.axhline(0, color="#c0c8e0", linewidth=0.6, linestyle="-")

for bar, val in zip(bars, resp_corr.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            val + (0.005 if val >= 0 else -0.012),
            f"{val:.3f}", ha="center", va="bottom" if val >= 0 else "top",
            fontsize=8, color="#c0c8e0", fontweight="600")

ax.set_title("Corrélation de chaque variable avec la réponse à la campagne (Response)", fontsize=12)
ax.set_ylabel("Corrélation de Pearson")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig("plots/05_correlations_target.png", bbox_inches="tight", dpi=150)
plt.show()

print("\n🔍 Top 5 variables les plus corrélées à Response :")
print(resp_corr.head().to_string())


In [ ]:
# ── Scatter plot : Income vs TotalSpend (coloré par Response) ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Relations entre variables numériques", fontsize=13, fontweight="600")

# Plot 1 : Income vs TotalSpend
for resp, color, label in [(0, PALETTE[3], "Non converti"), (1, PALETTE[0], "Converti")]:
    sub = df_clean[df_clean["Response"] == resp]
    axes[0].scatter(sub["Income"], sub["TotalSpend"], c=color, alpha=0.45,
                    s=18, label=label, edgecolors="none")
axes[0].set_title("Revenu vs Dépense totale")
axes[0].set_xlabel("Income (€)"); axes[0].set_ylabel("TotalSpend (€)")
axes[0].legend()

# Plot 2 : Recency vs TotalSpend
for resp, color, label in [(0, PALETTE[3], "Non converti"), (1, PALETTE[0], "Converti")]:
    sub = df_clean[df_clean["Response"] == resp]
    axes[1].scatter(sub["Recency"], sub["TotalSpend"], c=color, alpha=0.45,
                    s=18, label=label, edgecolors="none")
axes[1].set_title("Récence vs Dépense totale")
axes[1].set_xlabel("Recency (jours)"); axes[1].set_ylabel("TotalSpend (€)")
axes[1].legend()

plt.tight_layout()
plt.savefig("plots/06_scatter_plots.png", bbox_inches="tight", dpi=150)
plt.show()


In [ ]:
# ── Dépenses par catégorie × Éducation ─────────────────────────────────────────
spend_cols  = ["MntWines","MntFruits","MntMeatProducts",
               "MntFishProducts","MntSweetProducts","MntGoldProds"]
spend_names = ["Vins","Fruits","Viandes","Poissons","Sucreries","Or/Luxe"]

spend_edu = df_clean.groupby("Education")[spend_cols].mean()

fig, ax = plt.subplots(figsize=(13, 6))
x  = np.arange(len(spend_names))
bw = 0.15

for i, (edu, row) in enumerate(spend_edu.iterrows()):
    ax.bar(x + i*bw, row.values, bw, label=edu,
           color=PALETTE[i], alpha=0.85, edgecolor="#0d0f1a")

ax.set_xticks(x + bw * 2)
ax.set_xticklabels(spend_names, rotation=15, ha="right")
ax.set_title("Dépenses moyennes par catégorie de produit et niveau d'éducation")
ax.set_ylabel("Montant moyen (€)")
ax.legend(title="Éducation", fontsize=9)

plt.tight_layout()
plt.savefig("plots/07_depenses_education.png", bbox_inches="tight", dpi=150)
plt.show()


## 5. Analyse des Segments Clients <a id='5'></a>

> Identification des groupes les plus rentables et les plus réceptifs aux campagnes.

In [ ]:
# ── Taux de conversion par segment ─────────────────────────────────────────────
seg_vars = ["Education","Marital_Clean","Age_Group","Income_Group"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Taux de conversion (%) et dépense moyenne par segment", fontsize=14, fontweight="600")

for ax, seg in zip(axes.flat, seg_vars):
    seg_df = df_clean.groupby(seg).agg(
        Taux=("Response","mean"),
        Depense=("TotalSpend","mean")
    ).reset_index().sort_values("Taux", ascending=False)

    seg_df["Taux"] *= 100

    ax2 = ax.twinx()
    bars = ax.bar(seg_df[seg].astype(str), seg_df["Taux"],
                  color=PALETTE[0], alpha=0.85, edgecolor="#0d0f1a", label="Taux conv. (%)")
    ax2.plot(seg_df[seg].astype(str), seg_df["Depense"],
             color=PALETTE[2], marker="o", linewidth=2, markersize=7, label="Dépense moy. (€)")

    for bar, val in zip(bars, seg_df["Taux"]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                f"{val:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="600")

    ax.set_title(f"Segment : {seg}")
    ax.set_ylabel("Taux de conversion (%)")
    ax2.set_ylabel("Dépense moyenne (€)", color=PALETTE[2])
    ax2.tick_params(colors=PALETTE[2])
    ax.tick_params(axis="x", rotation=20)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, fontsize=8, loc="upper right")

plt.tight_layout()
plt.savefig("plots/08_segments_conversion.png", bbox_inches="tight", dpi=150)
plt.show()


In [ ]:
# ── Tableau récapitulatif des segments ─────────────────────────────────────────
summary_seg = df_clean.groupby("Income_Group").agg(
    Nb_clients      = ("Response","count"),
    Taux_conversion = ("Response","mean"),
    Revenu_moyen    = ("Income","mean"),
    Depense_totale  = ("TotalSpend","mean"),
    Nb_campagnes    = ("TotalCampaigns","mean"),
    Recence_moy     = ("Recency","mean"),
).round(2)

summary_seg["Taux_conversion"] = (summary_seg["Taux_conversion"]*100).round(1).astype(str) + "%"
summary_seg["Revenu_moyen"]    = summary_seg["Revenu_moyen"].apply(lambda x: f"{x:,.0f} €")
summary_seg["Depense_totale"]  = summary_seg["Depense_totale"].apply(lambda x: f"{x:,.0f} €")

print("📊 Profil détaillé par groupe de revenu :")
summary_seg


## 6. Performance des Campagnes <a id='6'></a>

In [ ]:
# ── Taux d'acceptation par campagne ────────────────────────────────────────────
camp_cols  = ["AcceptedCmp1","AcceptedCmp2","AcceptedCmp3",
              "AcceptedCmp4","AcceptedCmp5","Response"]
camp_names = ["Campagne 1","Campagne 2","Campagne 3",
              "Campagne 4","Campagne 5","Dernière campagne"]
camp_rates = df_clean[camp_cols].mean() * 100

fig, ax = plt.subplots(figsize=(11, 5))
colors_camp = PALETTE[:6]
bars = ax.bar(camp_names, camp_rates.values, color=colors_camp,
              edgecolor="#0d0f1a", alpha=0.88)

for bar, val in zip(bars, camp_rates.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.15,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="600")

ax.set_title("Taux d'acceptation par campagne marketing (%)")
ax.set_ylabel("Taux d'acceptation (%)")
ax.set_ylim(0, camp_rates.max() * 1.2)
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig("plots/09_performance_campagnes.png", bbox_inches="tight", dpi=150)
plt.show()


In [ ]:
# ── Canaux d'achat : convertis vs non-convertis ─────────────────────────────────
channel_cols  = ["NumWebPurchases","NumCatalogPurchases",
                 "NumStorePurchases","NumDealsPurchases"]
channel_names = ["Web","Catalogue","Magasin","Promotions"]

chan_resp = df_clean.groupby("Response")[channel_cols].mean()
chan_resp.columns = channel_names

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Comportement d'achat par canal", fontsize=13, fontweight="600")

# Camembert global
total_chan = df_clean[channel_cols].sum()
total_chan.index = channel_names
axes[0].pie(total_chan.values, labels=channel_names,
            colors=PALETTE[:4], autopct="%1.1f%%", startangle=120,
            wedgeprops=dict(linewidth=1.5, edgecolor="#0d0f1a"))
axes[0].set_title("Part de chaque canal (global)")

# Barres groupées par réponse
x  = np.arange(len(channel_names)); bw = 0.35
axes[1].bar(x - bw/2, chan_resp.loc[0], bw, label="Non-convertis",
            color=PALETTE[3], alpha=0.85, edgecolor="#0d0f1a")
axes[1].bar(x + bw/2, chan_resp.loc[1], bw, label="Convertis",
            color=PALETTE[0], alpha=0.85, edgecolor="#0d0f1a")
axes[1].set_xticks(x); axes[1].set_xticklabels(channel_names)
axes[1].set_title("Achats moyens par canal × Réponse")
axes[1].set_ylabel("Nombre d'achats moyen")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig("plots/10_canaux_achat.png", bbox_inches="tight", dpi=150)
plt.show()


## 7. Insights & Recommandations <a id='7'></a>

In [ ]:
# ── Synthèse automatique des insights ──────────────────────────────────────────
print("=" * 60)
print("  SYNTHÈSE — EDA MARKETING CAMPAIGN")
print("=" * 60)

conv_rate     = df_clean["Response"].mean() * 100
best_edu      = df_clean.groupby("Education")["Response"].mean().idxmax()
best_edu_rate = df_clean.groupby("Education")["Response"].mean().max() * 100
best_inc      = df_clean.groupby("Income_Group")["TotalSpend"].mean().idxmax()
best_age      = df_clean.groupby("Age_Group")["TotalSpend"].mean().idxmax()
top_canal     = pd.Series({
    "Web"      : df_clean["NumWebPurchases"].mean(),
    "Catalogue": df_clean["NumCatalogPurchases"].mean(),
    "Magasin"  : df_clean["NumStorePurchases"].mean(),
}).idxmax()
no_child_spend = df_clean[df_clean["HasChildren"]==0]["TotalSpend"].mean()
child_spend    = df_clean[df_clean["HasChildren"]==1]["TotalSpend"].mean()

insights = [
    ("Taux de conversion global",
     f"{conv_rate:.2f}% — 1 client sur {int(100/conv_rate)} accepte la campagne"),
    ("Meilleur segment éducation",
     f"Niveau '{best_edu}' → taux de conversion : {best_edu_rate:.1f}%"),
    ("Revenu & dépenses",
     f"Le segment '{best_inc}' dépense le plus en moyenne"),
    ("Tranche d'âge la plus dépensière",
     f"Groupe '{best_age}' → panier moyen le plus élevé"),
    ("Canal dominant",
     f"Le canal '{top_canal}' concentre le plus d'achats"),
    ("Impact des enfants",
     f"Sans enfants : {no_child_spend:.0f}€ moy. vs {child_spend:.0f}€ avec enfants"),
    ("Variable cible la plus corrélée",
     "TotalCampaigns et TotalSpend sont les prédicteurs les plus forts"),
]

for title, detail in insights:
    print(f"\n→ {title}")
    print(f"   {detail}")

print("\n" + "=" * 60)


In [ ]:
# ── Export du dataset nettoyé ───────────────────────────────────────────────────
df_clean.to_csv("marketing_campaign_clean.csv", index=False)
print("✅ Dataset nettoyé exporté : marketing_campaign_clean.csv")
print(f"   Shape finale : {df_clean.shape[0]} lignes × {df_clean.shape[1]} colonnes")


---
## Prochaines étapes

| Étape | Description |
|---|---|
| **Clustering K-Means** | Segmentation non supervisée des clients |
| **Modèle prédictif** | Random Forest pour prédire `Response` |
| **Analyse RFM** | Récence · Fréquence · Montant |
| **Test A/B** | Valider les segments identifiés |
| **Dashboard** | Application Streamlit déployée sur Streamlit Cloud |

---
*Notebook réalisé dans le cadre du Projet 1 — EDA · Data Science Portfolio*
